# 04 Modeling Data Split

Create reusable chronological training, validation, and test datasets from the cleaned electricity data.

- **Training:** January 1 through September 12, 2016
- **Validation:** September 13 through November 6, 2016
- **Testing:** November 7 through December 31, 2016

All rows are retained. Flat, zero, and incomplete building-days are not removed. The split is performed by calendar date to prevent a building-day from appearing in two datasets.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
source_path = project_root / "data" / "processed" / "electricity_cleaned.csv"
output_dir = project_root / "data" / "processed" / "modeling"
output_dir.mkdir(parents=True, exist_ok=True)

split_paths = {
    "train": output_dir / "model_train.parquet",
    "validation": output_dir / "model_validation.parquet",
    "test": output_dir / "model_test.parquet",
}

TRAIN_END = pd.Timestamp("2016-09-12 23:59:59")
VALIDATION_END = pd.Timestamp("2016-11-06 23:59:59")
CHUNK_ROWS = 500_000

source_path, split_paths

(PosixPath('/Users/srs/Desktop/PROJECTS/commercial-building-energy-prediction/data/processed/electricity_cleaned.csv'),
 {'train': PosixPath('/Users/srs/Desktop/PROJECTS/commercial-building-energy-prediction/data/processed/modeling/model_train.parquet'),
  'validation': PosixPath('/Users/srs/Desktop/PROJECTS/commercial-building-energy-prediction/data/processed/modeling/model_validation.parquet'),
  'test': PosixPath('/Users/srs/Desktop/PROJECTS/commercial-building-energy-prediction/data/processed/modeling/model_test.parquet')})

## Create the Chronological Splits

In [2]:
dtypes = {
    "building_id": "int16",
    "meter": "int8",
    "meter_reading": "float32",
    "site_id": "int8",
    "square_feet": "int32",
    "year_built": "float32",
    "air_temperature": "float32",
    "cloud_coverage": "float32",
    "dew_temperature": "float32",
    "precip_depth_1_hr": "float32",
    "sea_level_pressure": "float32",
    "wind_direction": "float32",
    "wind_speed": "float32",
    "hour": "int8",
    "day_of_week": "int8",
    "month": "int8",
    "is_weekend": "int8",
}

for path in split_paths.values():
    if path.exists():
        path.unlink()

writers = {name: None for name in split_paths}
summary = {
    name: {
        "rows": 0,
        "start": None,
        "end": None,
        "building_ids": set(),
    }
    for name in split_paths
}
source_rows = 0

try:
    for chunk in pd.read_csv(
        source_path,
        dtype=dtypes,
        parse_dates=["timestamp", "date"],
        chunksize=CHUNK_ROWS,
    ):
        source_rows += len(chunk)
        split_labels = np.select(
            [
                chunk["timestamp"] <= TRAIN_END,
                chunk["timestamp"] <= VALIDATION_END,
            ],
            ["train", "validation"],
            default="test",
        )

        for split_name, path in split_paths.items():
            split_chunk = chunk.loc[split_labels == split_name]
            if split_chunk.empty:
                continue

            table = pa.Table.from_pandas(split_chunk, preserve_index=False)
            if writers[split_name] is None:
                writers[split_name] = pq.ParquetWriter(
                    path,
                    table.schema,
                    compression="snappy",
                )
            writers[split_name].write_table(table)

            details = summary[split_name]
            details["rows"] += len(split_chunk)
            chunk_start = split_chunk["timestamp"].min()
            chunk_end = split_chunk["timestamp"].max()
            details["start"] = (
                chunk_start
                if details["start"] is None
                else min(details["start"], chunk_start)
            )
            details["end"] = (
                chunk_end
                if details["end"] is None
                else max(details["end"], chunk_end)
            )
            details["building_ids"].update(
                split_chunk["building_id"].unique().tolist()
            )
finally:
    for writer in writers.values():
        if writer is not None:
            writer.close()

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": details["rows"],
            "percent": details["rows"] / source_rows * 100,
            "buildings": len(details["building_ids"]),
            "start": details["start"],
            "end": details["end"],
            "path": split_paths[split_name],
        }
        for split_name, details in summary.items()
    ]
)

assert split_summary["rows"].sum() == source_rows
split_summary

,split,rows,percent,buildings,start,end,path
0,train,8395591,69.60993,1411,2016-01-01,2016-09-12 23:00:00,/Users/srs/Desktop/PROJECTS/commercial-buildin...
1,validation,1843685,15.28645,1408,2016-09-13,2016-11-06 23:00:00,/Users/srs/Desktop/PROJECTS/commercial-buildin...
2,test,1821634,15.10362,1407,2016-11-07,2016-12-31 23:00:00,/Users/srs/Desktop/PROJECTS/commercial-buildin...


## Validate the Saved Files

In [3]:
saved_summary = []

for split_name, path in split_paths.items():
    saved = pd.read_parquet(
        path,
        columns=["building_id", "timestamp", "meter_reading"],
    )
    saved_summary.append(
        {
            "split": split_name,
            "rows": len(saved),
            "buildings": saved["building_id"].nunique(),
            "start": saved["timestamp"].min(),
            "end": saved["timestamp"].max(),
            "missing_target": saved["meter_reading"].isna().sum(),
        }
    )

saved_summary = pd.DataFrame(saved_summary)

assert saved_summary.loc[0, "end"] < saved_summary.loc[1, "start"]
assert saved_summary.loc[1, "end"] < saved_summary.loc[2, "start"]
assert saved_summary["rows"].sum() == source_rows

saved_summary

,split,rows,buildings,start,end,missing_target
0,train,8395591,1411,2016-01-01,2016-09-12 23:00:00,0
1,validation,1843685,1408,2016-09-13,2016-11-06 23:00:00,0
2,test,1821634,1407,2016-11-07,2016-12-31 23:00:00,0


## Modeling Rule

Fit preprocessing and models using the training dataset. Use validation results for model selection and tuning. Evaluate the chosen final model on the test dataset only once.